In [1]:
from __future__ import annotations
import json
from pathlib import Path
import numpy as np

from starter_kit import observation, run_episode, load_json, load_training_data, logits

In [2]:
import torch
import torch.nn.functional as F
import random

In [3]:
GRID_SIZE = 8
MAX_STEPS = 96
INPUT_DIM = 397
MOVES = {0: (1, 0), 1: (-1, 0), 2: (0, 1), 3: (0, -1)}

N = 6

In [4]:
data_dir = Path("data")
training = load_training_data(data_dir)
X = training['features']
mask = training['action_mask']
y = training['actions']
kind = training["demo_kind"]
offsets = training["offsets"]

print(X.shape, mask.shape, y.shape, kind.shape, offsets.shape)

n_normal = 0
n_recovery = 0
for i in range(len(kind)):
    if kind[i] == 0:
        n_normal += 1
    else:
        n_recovery += 1
print("normal samples:", n_normal, "recovery samples:", n_recovery)

names = ["south", "north", "east", "west", "pickup", "dropoff"]
counts = [0] * N
for i in range(len(y)):
    counts[y[i]] += 1
for a in range(N):
    print(names[a], counts[a])

lengths = []
for d in range(len(offsets) - 1):
    lengths.append(offsets[d + 1] - offsets[d])
print("runs:", len(lengths), "shortest:", min(lengths), "longest:", max(lengths))

(4611, 397) (4611, 6) (4611,) (4611,) (361,)
normal samples: 4116 recovery samples: 495
south 945
north 939
east 959
west 1055
pickup 353
dropoff 360
runs: 360 shortest: 3 longest: 27


In [5]:
demos = load_json(data_dir / "train_demonstrations.json")
val = load_json(data_dir / "validation_scenarios.json")
print(str(demos)[:1500])
print(val[0])

[{'kind': 'normal', 'recovery_cell': None, 'recovery_injection_after_step': None, 'recovery_origin_cell': None, 'sample_count': 10, 'sample_end': 10, 'sample_start': 0, 'scenario_id': 'train_0000'}, {'kind': 'normal', 'recovery_cell': None, 'recovery_injection_after_step': None, 'recovery_origin_cell': None, 'sample_count': 18, 'sample_end': 28, 'sample_start': 10, 'scenario_id': 'train_0001'}, {'kind': 'normal', 'recovery_cell': None, 'recovery_injection_after_step': None, 'recovery_origin_cell': None, 'sample_count': 13, 'sample_end': 41, 'sample_start': 28, 'scenario_id': 'train_0002'}, {'kind': 'normal', 'recovery_cell': None, 'recovery_injection_after_step': None, 'recovery_origin_cell': None, 'sample_count': 9, 'sample_end': 50, 'sample_start': 41, 'scenario_id': 'train_0003'}, {'kind': 'normal', 'recovery_cell': None, 'recovery_injection_after_step': None, 'recovery_origin_cell': None, 'sample_count': 18, 'sample_end': 68, 'sample_start': 50, 'scenario_id': 'train_0004'}, {'kind

In [6]:
def drawgrid(scenario):
    walls = []
    for w in scenario["walls"]:
        walls.append((w[0], w[1]))
    depots = []
    for d in scenario["depots"]:
        depots.append((d[0], d[1]))
    package = depots[scenario["package_location"]]
    destination = depots[scenario["destination"]]
    start = (scenario["agent_pos"][0], scenario["agent_pos"][1])
    for r in range(GRID_SIZE):
        line = ""
        for c in range(GRID_SIZE):
            if (r, c) in walls:
                line += "#"
            elif (r, c) == start:
                line += "R"
            elif (r, c) == package:
                line += "P"
            elif (r, c) == destination:
                line += "D"
            elif (r, c) in depots:
                line += "o"
            else:
                line += "."
        print(line)

drawgrid(val[3])

..#.R...
......#.
...o.o.#
.....#..
.#...P#.
D.o#.#..
......o.
........


In [23]:
SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)

x = torch.from_numpy(X_aug)
y = torch.from_numpy(y_aug)
model = torch.nn.Sequential(
    torch.nn.Linear(397, 48),
    torch.nn.ReLU(),
    torch.nn.Linear(48, 32),
    torch.nn.ReLU(),
    torch.nn.Linear(32, 6),
)
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
weights = torch.tensor([1.0, 1.0, 1.0, 1.0, 2.5, 2.5])
generator = torch.Generator().manual_seed(SEED)

epochs = 30
for epoch in range(epochs):
    order = torch.randperm(len(x), generator=generator)
    total = 0.0
    for start in range(0, len(x), 256):
        batch = order[start : start + 256]
        loss = F.cross_entropy(
            model(x[batch]), y[batch], weight=weights
        )
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += float(loss.detach()) * len(batch)
    if epoch == 0 or (epoch + 1) % 5 == 0:
        print(f"epoch {epoch + 1:02d} loss={total / len(x):.4f}")

layers = [layer for layer in model if isinstance(layer, torch.nn.Linear)]
policy = {
    "W1": layers[0].weight.detach().numpy().T.astype(np.float32),
    "b1": layers[0].bias.detach().numpy().astype(np.float32),
    "W2": layers[1].weight.detach().numpy().T.astype(np.float32),
    "b2": layers[1].bias.detach().numpy().astype(np.float32),
    "W3": layers[2].weight.detach().numpy().T.astype(np.float32),
    "b3": layers[2].bias.detach().numpy().astype(np.float32),
}
validation = load_json(data_dir / "validation_scenarios.json")
results = [run_episode(scenario, policy) for scenario in validation]
success = np.mean([result["success"] for result in results])
print(f"local validation success={success:.4f}")

epoch 01 loss=1.1808
epoch 05 loss=0.3074
epoch 10 loss=0.2295
epoch 15 loss=0.2039
epoch 20 loss=0.1763
epoch 25 loss=0.1575
epoch 30 loss=0.1422
local validation success=0.6979


In [21]:
correct = 0
for i in range(len(X)):
    scores = logits(X[i], policy)
    best = -1
    for a in range(6):
        if mask[i][a] and (best == -1 or scores[a] > scores[best]):
            best = a
    if best == training["actions"][i]:
        correct += 1
print("training accuracy per step:", correct / len(X))

training accuracy per step: 0.9262632834526133


In [9]:
def run_with_loops(scenario, policy):
    position = tuple(scenario["agent_pos"])
    carrying = False
    walls = {tuple(value) for value in scenario["walls"]}
    depots = [tuple(value) for value in scenario["depots"]]
    package = depots[scenario["package_location"]]
    destination = depots[scenario["destination"]]
    visited = set()                            
    for step in range(MAX_STEPS):
        if (position, carrying) in visited:     
            return "loop"                      
        visited.add((position, carrying))      
        features, mask = observation(scenario, position, carrying)
        scores = np.where(mask, logits(features, policy), -np.inf)
        action = int(np.argmax(scores))
        if action in MOVES:
            dr, dc = MOVES[action]
            nxt = (position[0] + dr, position[1] + dc)
            if nxt not in walls:
                position = nxt
        elif action == 4 and not carrying and position == package:
            carrying = True
        elif action == 5 and carrying and position == destination:
            return "success"
    return "timeout"

In [10]:
train_scen = load_json(data_dir / "training_scenarios.json")
for name, scenarios in [("training maps", train_scen), ("validation maps", val)]:
    result = {"success": 0, "loop": 0, "timeout": 0}
    for s in scenarios:
        result[run_with_loops(s, policy)] += 1
    print(name, result)

training maps {'success': 299, 'loop': 61, 'timeout': 0}
validation maps {'success': 37, 'loop': 59, 'timeout': 0}


In [12]:
def trace(scenario, policy):
    position = tuple(scenario["agent_pos"])
    carrying = False
    walls = {tuple(value) for value in scenario["walls"]}
    depots = [tuple(value) for value in scenario["depots"]]
    package = depots[scenario["package_location"]]
    destination = depots[scenario["destination"]]
    visited = set()
    path = []                                            # new
    for step in range(MAX_STEPS):
        if (position, carrying) in visited:
            return "loop", path, position, carrying      # new
        visited.add((position, carrying))
        path.append(position)                            # new
        features, mask = observation(scenario, position, carrying)
        scores = np.where(mask, logits(features, policy), -np.inf)
        action = int(np.argmax(scores))
        if action in MOVES:
            dr, dc = MOVES[action]
            nxt = (position[0] + dr, position[1] + dc)
            if nxt not in walls:
                position = nxt
        elif action == 4 and not carrying and position == package:
            carrying = True
        elif action == 5 and carrying and position == destination:
            return "success", path, position, carrying   # new
    return "timeout", path, position, carrying           # new

def drawpath(scenario, path, loop_cell):
    walls = []
    for w in scenario["walls"]:
        walls.append((w[0], w[1]))
    depots = []
    for d in scenario["depots"]:
        depots.append((d[0], d[1]))
    package = depots[scenario["package_location"]]
    destination = depots[scenario["destination"]]
    start = (scenario["agent_pos"][0], scenario["agent_pos"][1])
    for r in range(GRID_SIZE):
        line = ""
        for c in range(GRID_SIZE):
            if (r, c) in walls:
                line += "#"
            elif (r, c) == loop_cell:
                line += "X"
            elif (r, c) == start:
                line += "R"
            elif (r, c) == package:
                line += "P"
            elif (r, c) == destination:
                line += "D"
            elif (r, c) in depots:
                line += "o"
            elif (r, c) in path:
                line += "*"
            else:
                line += "."
        print(line)

shown = 0
for s in val:
    outcome, path, last, carrying = trace(s, policy)
    if outcome == "loop" and shown < 4:
        print(s["scenario_id"], "carrying:", carrying)
        drawpath(s, path, last)
        print()
        shown += 1

validation_001 carrying: True
........
.##...D.
##o.##*X
#..P*R**
......o.
.#......
...o....
.o......

validation_004 carrying: False
..#..R..
...##*..
....PX.#
.....*.o
..#D....
.......o
.o...##o
......#.

validation_005 carrying: True
..#oo...
........
.R...o..
**.#o..#
P.....#.
*..#.#..
*#..*...
****X#D.

validation_006 carrying: False
...#.oo#
......#o
D..#***R
...#X*o#
..P#....
........
..#.....
........



## transformari pe grid

In [14]:
def rotate_cell(cell):
    return (cell[1], GRID_SIZE-1-cell[0])

def mirror_cell(cell):
    return (cell[0], GRID_SIZE-1-cell[1])

def transform_cell(cell, k, flip):
    r, c = cell[0], cell[1]
    if flip:
        r, c = mirror_cell((r, c))
    for i in range(k):
        r, c = rotate_cell((r, c))
    return (r, c)

def transform_scenario(scenario, k, flip):
    new = dict(scenario)
    walls = []
    for w in scenario["walls"]:
        walls.append(list(transform_cell(w, k, flip)))
    depots = []
    for d in scenario["depots"]:
        depots.append(list(transform_cell(d, k, flip)))
    new["walls"] = walls
    new["depots"] = depots
    new["agent_pos"] = list(transform_cell(scenario["agent_pos"], k, flip))
    return new

def transform_action(action, k, flip):
    if action not in MOVES:
        return action                # pickup and dropoff don't change
    dr, dc = MOVES[action]
    if flip:
        dc = -dc
    for i in range(k):
        dr, dc = dc, -dr             # a move turns the same way a cell does
    for a in MOVES:
        if MOVES[a] == (dr, dc):
            return a

In [15]:
# validam transformarile

def replay(scenario, actions):
    position = tuple(scenario["agent_pos"])
    carrying = False
    for a in actions:
        features, legal = observation(scenario, position, carrying)
        if not legal[a]:
            return False
        if a in MOVES:
            dr, dc = MOVES[a]
            position = (position[0] + dr, position[1] + dc)
        elif a == 4:
            carrying = True
        elif a == 5:
            return True
    return False

by_id = {}
for s in train_scen:
    by_id[s["scenario_id"]] = s

d = demos[0]
s = by_id[d["scenario_id"]]
acts = training["actions"][d["sample_start"]:d["sample_end"]]
for flip in [False, True]:
    for k in range(4):
        s2 = transform_scenario(s, k, flip)
        acts2 = []
        for a in acts:
            acts2.append(transform_action(int(a), k, flip))
        print("turns:", k, "mirror:", flip, "expert delivers:", replay(s2, acts2))
        drawgrid(s2)
        print()

turns: 0 mirror: False expert delivers: True
........
..#..Po.
......#.
...#.D..
o...#...
......#.
..#.R.#.
o#o.....

turns: 1 mirror: False expert delivers: True
o..o....
#.......
o#....#.
....#...
.R.#....
....D.P.
.##..#o.
........

turns: 2 mirror: False expert delivers: True
.....o#o
.#.R.#..
.#......
...#...o
..D.#...
.#......
.oP..#..
........

turns: 3 mirror: False expert delivers: True
........
.o#..##.
.P.D....
....#.R.
...#....
.#....#o
.......#
....o..o

turns: 0 mirror: True expert delivers: True
........
.oP..#..
.#......
..D.#...
...#...o
.#......
.#.R.#..
.....o#o

turns: 1 mirror: True expert delivers: True
........
.##..#o.
....D.P.
.R.#....
....#...
o#....#.
#.......
o..o....

turns: 2 mirror: True expert delivers: True
o#o.....
..#.R.#.
......#.
o...#...
...#.D..
......#.
..#..Po.
........

turns: 3 mirror: True expert delivers: True
....o..o
.......#
.#....#o
...#....
....#.R.
.P.D....
.o#..##.
........



In [25]:
def all_starts(scenario):
    walls = []
    for w in scenario["walls"]:
        walls.append((w[0], w[1]))
    starts = []
    for r in range(GRID_SIZE):
        for c in range(GRID_SIZE):
            if (r, c) not in walls:
                s = dict(scenario)
                s["agent_pos"] = [r, c]
                starts.append(s)
    return starts

def hard_test(policy, scenarios, stall_prob):
    rng = np.random.default_rng(0)   # same stalls for every network we compare
    ok = 0
    total = 0
    for s in scenarios:
        for s2 in all_starts(s):
            stalls = set()
            for step in range(MAX_STEPS):
                if rng.random() < stall_prob:
                    stalls.add(step)
            if run_episode(s2, policy, stalls=stalls)["success"]:
                ok += 1
            total += 1
    return ok / total

print("every start, no stalls: ", hard_test(policy, val, 0.0))
print("every start, 20% stalls:", hard_test(policy, val, 0.2))

every start, no stalls:  0.7079613095238095
every start, 20% stalls: 0.7079613095238095


In [16]:
X_aug = []
y_aug = []
m_aug = []
mismatches = 0
for d in demos:
    s = by_id[d["scenario_id"]]
    for flip in [False, True]:
        for k in range(4):
            s2 = transform_scenario(s, k, flip)
            for i in range(d["sample_start"], d["sample_end"]):
                r = int(round(X[i][384] * 7))
                c = int(round(X[i][385] * 7))
                carrying = bool(X[i][388] > 0.5)
                pos2 = transform_cell((r, c), k, flip)
                features, legal = observation(s2, pos2, carrying)
                if k == 0 and not flip and not np.array_equal(features, X[i]):
                    mismatches += 1
                X_aug.append(features)
                m_aug.append(legal)
                y_aug.append(transform_action(int(training["actions"][i]), k, flip))

X_aug = np.array(X_aug, dtype=np.float32)
m_aug = np.array(m_aug)
y_aug = np.array(y_aug, dtype=np.int64)
print(X_aug.shape, y_aug.shape, "mismatches:", mismatches)

(36888, 397) (36888,) mismatches: 0


In [17]:
x = torch.from_numpy(X_aug)
y = torch.from_numpy(y_aug)

In [24]:
from export_policy import export_policy
from validate_submission import validate

export_policy(policy, Path("submission.zip"))
validate(Path("submission.zip"))
print("submission is valid")

submission is valid
